# 01 · Data preprocessing

Launches the data step: `MILDataModule.setup()` reads the CSV, builds
**bags** (one bag = one molecule, instances = conformers), optionally clusters
conformers per molecule, fits a per-fingerprint-block `StandardScaler` on the
**train split only**, and produces the train/val/test datasets.

The split is predefined (`split` column: 0=train, 1=val, 2=test) — the same
`MILDataModule` is used for final training.

In [1]:
# --- Bootstrap: make the notebook run from anywhere ---
import os, sys, logging
from pathlib import Path

# Locate the project root (folder that contains the `ppl` package).
here = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [here, *here.parents] if (p / 'ppl' / '__init__.py').exists()),
    None,
)
if PROJECT_ROOT is None:
    # Fallback: this notebook lives in <root>/notebooks/
    PROJECT_ROOT = Path('__file__' in globals() and __file__ or '.').resolve().parent.parent

os.chdir(PROJECT_ROOT)                       # pipeline writes outputs relative to cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
print('Project root:', PROJECT_ROOT)

Project root: /Users/vfastovskii/Desktop/milkid


In [2]:
# Path to the experiment YAML. Edit this to point at a different config.
CONFIG_PATH = PROJECT_ROOT / 'ppl/config/experiment_configs/run_config.yaml'
assert CONFIG_PATH.exists(), f'Config not found: {CONFIG_PATH}'
print('Using config:', CONFIG_PATH.relative_to(PROJECT_ROOT))

Using config: ppl/config/experiment_configs/run_config.yaml


## Build the data config

We reuse `PipelineConfigManager` so the `DataLoaderConfig` is identical to what
the pipeline builds (defaults + YAML overrides + experiment-name/cache wiring).

In [3]:
from ppl.config.pipeline_config import PipelineConfig
from ppl.pipeline.config_manager import PipelineConfigManager

cfg = PipelineConfig.from_yaml(CONFIG_PATH)
data_cfg = PipelineConfigManager(cfg).data_cfg
print('CSV      :', data_cfg.csv_path)
print('task     :', data_cfg.task)
print('batch    :', data_cfg.batch_size)
print('clustering:', data_cfg.cluster_instances)

INFO ppl.config.pipeline_config: Using default csv_path: /Users/vfastovskii/Desktop/milkid/ppl/data/trypsin_usrcat.csv
INFO ppl.pipeline.config_override_utils: [EXP CFG] Applied overrides to DataLoaderConfig
INFO ppl.pipeline.config_manager: Using experiment_name 'bace809_cluster_hier_mha_200c_2jul_bs16_test15' for data splits
INFO ppl.pipeline.config_manager: Setting default cache_dir to 'bace809_cluster_hier_mha_200c_2jul_bs16_test15' for data splits
INFO ppl.pipeline.config_override_utils: [EXP CFG] Applied overrides to ModelBuilderConfig
INFO ppl.pipeline.config_override_utils: [EXP CFG] Applied overrides to TrainerOptimConfig
INFO ppl.pipeline.config_override_utils: [EXP CFG] Applied overrides to TrainerConfig
INFO ppl.pipeline.config_manager: Setting global seed to 42
INFO ppl.utils.reproducibility.deterministic_setup: Deterministic setup complete with seed: 42
INFO ppl.utils.reproducibility.deterministic_setup: CUBLAS_WORKSPACE_CONFIG = :4096:8
INFO ppl.utils.reproducibility.det

CSV      : ppl/data/bace809_3d_fp_5fp_200c_05prune_6kcal_no_Hs_june_concated_splited.csv
task     : regression
batch    : 16
clustering: True


## Run preprocessing

`setup()` does all the heavy lifting: it reads the CSV, applies the predefined
split (train=0, val=1, test=2), builds bags, clusters, and scales features.

In [4]:
from ppl.data.data_loader import MILDataModule

dm = MILDataModule(data_cfg)
dm.setup('fit')   # loads CSV, applies predefined split, builds bags, clusters, scales
print('setup complete')

INFO ppl.data.data_loader: [DM] Initialising MILDataModule with DataLoaderConfig.
INFO ppl.data.data_loader: [DM] Using cache directory: /Users/vfastovskii/Desktop/milkid/bace809_cluster_hier_mha_200c_2jul_bs16_test15/bace809_cluster_hier_mha_200c_2jul_bs16_test15
INFO ppl.data.data_module_impl: [DM] DataModule.setup(stage=fit) called
INFO ppl.data.data_module_impl: [DM] Initial memory usage: 0.48 GB
INFO ppl.data.data_module_impl: [DM] Train directory does not exist: /Users/vfastovskii/Desktop/milkid/bace809_cluster_hier_mha_200c_2jul_bs16_test15/bace809_cluster_hier_mha_200c_2jul_bs16_test15/train
INFO ppl.data.data_module_impl: [DM] Test directory does not exist: /Users/vfastovskii/Desktop/milkid/bace809_cluster_hier_mha_200c_2jul_bs16_test15/bace809_cluster_hier_mha_200c_2jul_bs16_test15/test
INFO ppl.data.data_module_impl: [DM] Val directory does not exist: /Users/vfastovskii/Desktop/milkid/bace809_cluster_hier_mha_200c_2jul_bs16_test15/bace809_cluster_hier_mha_200c_2jul_bs16_test

setup complete


## Inspect the result

In [5]:
n_feat = len(dm.feature_names)
print('num descriptors (input_dim):', n_feat)
print('first 8 feature names      :', dm.feature_names[:8])
print()
for name, ds in [('train', dm._train), ('val', dm._val), ('test', dm._test)]:
    n = 0 if ds is None else len(ds)
    print(f'{name:<5} bags: {n}')

num descriptors (input_dim): 472
first 8 feature names      : ['ElectroShapeFingerprint_0', 'ElectroShapeFingerprint_1', 'ElectroShapeFingerprint_2', 'ElectroShapeFingerprint_3', 'ElectroShapeFingerprint_4', 'ElectroShapeFingerprint_5', 'ElectroShapeFingerprint_6', 'ElectroShapeFingerprint_7']

train bags: 647
val   bags: 324
test  bags: 0


### Peek at one batch

The batch layout is `(bags, labels, bag_ids, padding_mask, cluster_ids, series_labels)`.
Shapes tell you the padded bag size and descriptor dimension.

In [6]:
train_loader = dm.train_dataloader()
batch = next(iter(train_loader))

def describe(x):
    import torch
    if isinstance(x, torch.Tensor):
        return f'Tensor{tuple(x.shape)} {x.dtype}'
    if isinstance(x, (list, tuple)):
        return f'{type(x).__name__}(len={len(x)})'
    return type(x).__name__

labels = ['bags', 'labels', 'bag_ids', 'padding_mask', 'cluster_ids', 'series_labels']
for i, part in enumerate(batch):
    name = labels[i] if i < len(labels) else f'item_{i}'
    print(f'{name:<14}: {describe(part)}')

INFO ppl.data.data_loader_impl: Using seeded DataLoader generator for shuffled training batches: seed=42
INFO ppl.data.data_loader_impl: [SERIES_BATCH] Using balanced train batch sampler: batch_size=16, series=13, batches_per_epoch=65, coverage=all_train_bags
INFO ppl.data.data_loader_impl: Created series-balanced DataLoader with batch_size=16, num_workers=10, pin_memory=False, persistent_workers=True, device=mps


bags          : Tensor(16, 199, 472) torch.float32
labels        : Tensor(16,) torch.float32
bag_ids       : list(len=16)
padding_mask  : Tensor(16, 199) torch.bool
cluster_ids   : Tensor(16, 199) torch.int64
series_labels : list(len=16)


### Label distribution

Quick sanity check on the endpoint values across splits.

In [7]:
import numpy as np
import torch

def collect_labels(loader):
    ys = []
    for b in loader:
        y = b[1]
        ys.append(y.detach().cpu().numpy().reshape(-1))
    return np.concatenate(ys) if ys else np.array([])

for name, loader in [('train', dm.train_dataloader()),
                     ('val', dm.val_dataloader()),
                     ('test', dm.test_dataloader())]:
    if loader is None or (hasattr(loader, '__len__') and len(loader) == 0):
        print(f'{name:<5}: (empty)')
        continue
    y = collect_labels(loader)
    if y.size:
        print(f'{name:<5}: n={y.size:<4} mean={y.mean():.3f} std={y.std():.3f} min={y.min():.3f} max={y.max():.3f}')

INFO ppl.data.data_loader_impl: Using seeded DataLoader generator for shuffled training batches: seed=42
INFO ppl.data.data_loader_impl: [SERIES_BATCH] Using balanced train batch sampler: batch_size=16, series=13, batches_per_epoch=65, coverage=all_train_bags
INFO ppl.data.data_loader_impl: Created series-balanced DataLoader with batch_size=16, num_workers=10, pin_memory=False, persistent_workers=True, device=mps
INFO ppl.data.data_loader_impl: Created DataLoader with batch_size=16, num_workers=10, pin_memory=False, persistent_workers=True, device=mps
INFO ppl.data.data_loader_impl: Created DataLoader with batch_size=16, num_workers=10, pin_memory=False, persistent_workers=True, device=mps


train: n=1040 mean=6.560 std=1.151 min=3.190 max=9.000
val  : n=324  mean=6.911 std=1.159 min=3.000 max=9.190
test : (empty)


---
The key output for the next step is **`input_dim = len(dm.feature_names)`**.
Continue with **`02_model_construction.ipynb`**.